In [1]:
import pandas as pd
import numpy as np


from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt
import pickle

import os

In [2]:
import xgboost as xgb
import dill #aneto
import SuperLore
import category_encoders
import re
import json

## Controregole-controfattuali

#### Carico il Modello

In [3]:
bb = xgb.XGBClassifier()
bb.load_model("../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_train.model")

In [4]:
bb

XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=0.65, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.025, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=13, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=110, n_jobs=10, num_parallel_tree=None,
              objective='binary:hinge', predictor=None, ...)

#### Caricol x train, y train, x test, y test

In [5]:
X_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtrain')
Y_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytrain')
X_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtest')
Y_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytest')
data_desc=pd.read_pickle('../datasets/Dati-Banca-Lore/intesa_incassi_data_description.p')

#### Carico le spiegazioni di Lore

In [6]:
lore_exp_path=open('../datasets/Dati-Banca-Lore/lore_exp_INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xgb_cfs_binary_from_dts.p','rb')
objs = []
while 1:
    try:
        objs.append(pickle.load(lore_exp_path))
    except EOFError:
        break

### Le regole: si esportano come una lista di dizionari. Ogni dizionario è un predicato della regola

In [7]:
for lore_p in objs[1].rule.premises:
    print(lore_p.op)
    print(lore_p.att)
    print(lore_p.is_continuous)
   

=
PN_SEPA_USCITA_TY_NUM
False
=
PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM
False
=
SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE
False


In [8]:
rule_list=[]
for lore_r in objs[1].rule.premises:
    rule_list.append(vars(lore_r))    
print(rule_list)

[{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}]


### Guardo gli exemplars

In [9]:
objs[1].exemplars

'{ PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO = 0.4654900164207794PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO = 0.17967654780246395PRODV_LETTERE_DI_CREDITO_PON = 0.45295295295295296PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM = 0.7772772772772774PN_SEPA_ENTRATA_TY_VAL = 0.5551001447826013SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE = 0.24599321712265734PN_SEPA_USCITA_PRC_DLT_YEAR_VAL = 0.4548162197285004PCRIV_FT_20_DLT_ANNUO_UTILZZ_ACCRD_MEDIO = 0.43940362988916437PN_SEPA_USCITA_TY_NUM = 0.5989829183526338PRODV_FACTORING_PON = 0.4624624624624625PRODV_SECURITIZATION_PON = 0.4984984984984985BESTV_BON_VERSO_NON_EU_TY_VAL = 0.4155718243037367PCRIV_MERCATO_EUR_TY = 1.0PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL = 0.48979983431707 }\n{ PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO = 0.5592133889147981PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO = 0.3788947253156259PRODV_LETTERE_DI_CREDITO_PON = 0.45295295295295296PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM = 0.6206206206206206PN_SEPA_ENTRATA_TY_VAL = 0.5142732253469908SCADV_IMP_ESPOSIZIONE_ATTUAL

In [10]:
def parse_exemplars(item):
    ex=item.exemplars
    new_ex = re.sub(r'(\d\.\d+)', r'\1 ', ex)
    pattern = r'{(.*?)}'
    matches = re.findall(pattern, new_ex, re.DOTALL)
    exemplars_list = []
    for match in matches:
        ex_parse = {}
        list_to_dict=[]
        properties = match.split() #ogni stringa separata da uno spazio diventa un nuovo item della lista, ogni lista è un exemplar
        for prop in properties: 
            if prop != "=": #elimino =
                list_to_dict.append(prop) # creo un dizionario con chiave e valore
        for i in range(0, len(list_to_dict), 2):
            ex_parse[list_to_dict[i]] = float(list_to_dict[i + 1])
        exemplars_list.append(ex_parse)
    return exemplars_list

In [11]:
print(parse_exemplars(objs[1]))

[{'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': 0.4654900164207794, 'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': 0.17967654780246395, 'PRODV_LETTERE_DI_CREDITO_PON': 0.45295295295295296, 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': 0.7772772772772774, 'PN_SEPA_ENTRATA_TY_VAL': 0.5551001447826013, 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE': 0.24599321712265734, 'PN_SEPA_USCITA_PRC_DLT_YEAR_VAL': 0.4548162197285004, 'PCRIV_FT_20_DLT_ANNUO_UTILZZ_ACCRD_MEDIO': 0.43940362988916437, 'PN_SEPA_USCITA_TY_NUM': 0.5989829183526338, 'PRODV_FACTORING_PON': 0.4624624624624625, 'PRODV_SECURITIZATION_PON': 0.4984984984984985, 'BESTV_BON_VERSO_NON_EU_TY_VAL': 0.4155718243037367, 'PCRIV_MERCATO_EUR_TY': 1.0, 'PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL': 0.48979983431707}, {'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': 0.5592133889147981, 'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': 0.3788947253156259, 'PRODV_LETTERE_DI_CREDITO_PON': 0.45295295295295296, 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': 0.6206206206206206, 'PN_SEPA_ENTRATA_TY_VA

### Contro esemplari: queste istanze non hanno controesemplari

In [12]:
for lore_ce in objs:
    print(lore_ce.cexemplars)


None
None
None
None
None
None
None


In [13]:
for cr in objs[4].crules:
    print(type(cr))

<class 'SuperLore.rule.Rule'>


### Contro regole: ma sono uguali alle regole??

In [14]:
counterrule_list=[]
for obj in objs:
    for cr in objs[1].crules:
        for nested in cr.premises:
            counterrule_list.append(vars(nested))
        print(counterrule_list)

[{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}]
[{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}]
[{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZI

In [18]:
counterrule_list=[]
#for obj in objs:
for cr in objs[1].crules:
    for nested in cr.premises:
        counterrule_list.append(vars(nested))
    print(counterrule_list)

[{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}]


In [ ]:
X_train

In [24]:
print(vars(objs[1].rule))

{'premises': [<SuperLore.rule.Condition object at 0x131854350>, <SuperLore.rule.Condition object at 0x131854150>, <SuperLore.rule.Condition object at 0x131854110>], 'cons': 1, 'class_name': 'Target'}


#### ESPORTO IL JASON

In [ ]:
filename = "istance_1.json"
with open(filename, "w") as file:
    json.dump(data, file)